In [ ]:
import os
from openai import OpenAI
import threading
from dotenv import load_dotenv

In [ ]:
load_dotenv()
client = OpenAI(base_url = "https://openai.vocareum.com/v1", api_key=os.getenv(""))
agent_outputs = {}

In [ ]:
def call_openai(system_prompt , user_prompt):
  response = client.chat.completions(
      model="gpt-3.5-turbo",
      messages=[
          {"role":"system","content":syetem_prompt},
          {"role":"user","content":user_promp}
      ],
      temprature =0,
  )
  return response.choice[0].message.content

In [ ]:
contract_text = """
CONSULTING AGREEMENT

This Consulting Agreement (the "Agreement") is made effective as of January 1, 2025 (the "Effective Date"), by and between ABC Corporation, a Delaware corporation ("Client"), and XYZ Consulting LLC, a California limited liability company ("Consultant").

1. SERVICES. Consultant shall provide Client with the following services: strategic business consulting, market analysis, and technology implementation advice (the "Services").

2. TERM. This Agreement shall commence on the Effective Date and shall continue for a period of 12 months, unless earlier terminated.

3. COMPENSATION. Client shall pay Consultant a fee of $10,000 per month for Services rendered. Payment shall be made within 30 days of receipt of Consultant's invoice.

4. CONFIDENTIALITY. Consultant acknowledges that during the engagement, Consultant may have access to confidential information. Consultant agrees to maintain the confidentiality of all such information.

5. INTELLECTUAL PROPERTY. All materials developed by Consultant shall be the property of Client. Consultant assigns all right, title, and interest in such materials to Client.

6. TERMINATION. Either party may terminate this Agreement with 30 days' written notice. Client shall pay Consultant for Services performed through the termination date.

7. GOVERNING LAW. This Agreement shall be governed by the laws of the State of Delaware.

8. LIMITATION OF LIABILITY. Consultant's liability shall be limited to the amount of fees paid by Client under this Agreement.

9. INDEMNIFICATION. Client shall indemnify Consultant against all claims arising from use of materials provided by Client.

10. ENTIRE AGREEMENT. This Agreement constitutes the entire understanding between the parties and supersedes all prior agreements.

IN WITNESS WHEREOF, the parties have executed this Agreement as of the date first above written.
"""

In [ ]:
class LegalTermsChecker:
    """Agent that checks for problematic legal terms and clauses in contracts."""
    def run(self, contract_text):
      print("LegalTermsChecker: Analyzing contract for problematic legal terms...")
      system_prompt = "You are a legal expert specializing in contract law. Review the provided contract text and identify any problematic clauses, ambiguous terms, or non-standard legal language. List your key findings."
      user_prompt = """ f"Analyze: {contract_text}"""
      agent_outputs["legal"]=call_openai(system_prompt, user_prompt)

In [ ]:
class ComplianceValidator:
    """Agent that validates regulatory and industry compliance of contracts."""
    def run(self, contract_text):
        print("ComplianceValidator : Validate Regulatory and industry compliance of contracts...")
        system_prompt = "You are a Compliance validating expert specializing in contract law. Review the provided contract text and you have to work by checking, testing, and documenting that a company's systems, data, or processes adhere to specific regulatory standards, industry benchmarks, or internal policies"
        user_prompt = """ f"Analyze: {contract_text}"""
        agent_outputs["compliance"]= call_openai(system_prompt, user_prompt)

In [ ]:
class FinancialRiskAssessor:
    """Agent that assesses financial risks and liabilities in contracts."""
    def run(self, contract_text):
        print("FinancialRiskAssessor: Assess financial risks and liabilities in contracts...")
        system_prompt = "You are a financial risk assessment expert specializing in contract law. Review the contract and analyse the contract to find financial risks in the contract"""
        user_prompt = """ f"Analyze: {contract_text}"""
        agent_outputs["financial"] =call_openai(system_prompt, user_prompt)

In [ ]:
class SummaryAgent:
    """Agent that synthesizes findings from all specialized agents."""
    def run(self, contract_text, inputs):
      print("SummaryAgent: Synthesizing all findings...")
      legal_findings = inputs.get("legal", "No legal analysis provided.")
      compliance_findings = inputs.get("compliance", "No compliance analysis provided.")
      financial_findings = inputs.get("financial", "No financial analysis provided.")
      system_prompt = "You are a senior legal counsel..." # As described above
      user_prompt = f"""Please synthesize the following analyses of a contract into a comprehensive summary report.
      Original Contract Text (for reference, if needed, but focus on the analyses):
      --- BEGIN CONTRACT TEXT (abbreviated for prompt, or just mention it was analyzed) ---
      {contract_text[:500]}...
      --- END CONTRACT TEXT ---
      Legal Terms Analysis:
      {legal_findings}
      Compliance Validation:
      {compliance_findings}
      Financial Risk Assessment:
      {financial_findings}
      Provide a consolidated executive summary identifying key issues and an overall assessment.
      """
      return call_openai(system_prompt, user_prompt)


In [ ]:
def analyze_contract(contract_text):
    """Run all agents in parallel and summarize their findings."""
    legal_checker = LegalTermsChecker()
    compliance_validator = ComplianceValidator()
    financial_risk_assessor = FinancialRiskAssessor()
    # Run specialist agents in parallel using threading
    threads = [
    threading.Thread(target=legal_checker.run, args=(contract_text,)),
    threading.Thread(target=compliance_validator.run, args=(contract_text,)),
    threading.Thread(target=financial_risk_assessor.run, args=(contract_text,))
      ]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    final_analysis = SummaryAgent.run(contract_text, agent_outputs)

    return final_analysis

In [ ]:
print("Enterprise Contract Analysis System")
print("Analyzing contract...")

# TODO: Call the analyze_contract function and print results
final_analysis = analyze_contract(contract_text)
print("\n=== FINAL CONTRACT ANALYSIS ===\n")
print(final_analysis)